# Utils Notebook from Álvaro

---
## `cast_list_as_strings(list)`

This function has been directly obtained from the provided jupyter notebook in order to ensure all questions on the data were of type string. No further modifications have been made.

In [ ]:
def cast_list_as_strings(mylist):
    """
    return a list of strings
    """
    mylist_of_strings = []
    for x in mylist:
        mylist_of_strings.append(str(x))

    return mylist_of_strings

---
## `get_interaction_features_from_df(df, count_vectorizer)`

This function uses the trained Count Vectorizer to transform the questions into numerical matrices. Then, we substract and multiply the matrices so that the model can learn from combined numerical data instead of having separate information for each question. In the comments we try to explain a little further what we expect from each operation.

This function was also present on the provided jupyter notebook. However, in this case we changed it by stacking these transformations instead of stacking the independent matrices of each question.

In [ ]:
def get_interaction_features_from_df(df, count_vectorizer):
    """
    Returns a sparse matrix containing the interaction features built by the count vectorizer.
    Instead of horizontally stacking the features of question1 and question2 independently, 
    it computes the absolute difference and the element-wise product of their sparse matrices.
    This provides symmetric features and allows linear models to capture specific word overlaps.
    """
    q1_casted = cast_list_as_strings(list(df["question1"]))
    q2_casted = cast_list_as_strings(list(df["question2"]))

    # Transform text into sparse Bag-of-Words matrices
    X_q1 = count_vectorizer.transform(q1_casted)
    X_q2 = count_vectorizer.transform(q2_casted)    
    
    # 1. Absolute Difference: |X_q1 - X_q2|
    # If a word is present in both, 1 - 1 = 0
    # If a word is present in only one of them, |1 - 0| = 1 or |0 - 1| = 1
    # In this way, we expect the linear model to learn negative weights for words that do not match
    X_diff = abs(X_q1 - X_q2)
    
    # 2. Element-wise Product: X_q1 * X_q2
    # If a word is present in both, 1 * 1 = 1
    # If it is missing in one of them, 1 * 0 = 0
    # Like this, we want the linear model to learn which specific shared words are key to predict duplicates
    X_prod = X_q1.multiply(X_q2)
    
    # Horizontally stack the interaction matrices instead of the independent vectors
    X_interactions = scipy.sparse.hstack((X_diff, X_prod))

    return X_interactions

---
## `get_mistakes(classifier, csr_matrix, numpy_array)`

This function has been directly obtained from the provided jupyter notebook in order to obtain all the mistakes that the classifier makes. No further modifications have been made.

In [ ]:
def get_mistakes(clf, X_q1q2, y):
    ############### Begin exercise ###################
    predictions = clf.predict(X_q1q2)
    incorrect_predictions = predictions != y 
    incorrect_indices,  = np.where(incorrect_predictions)
    ############### End exercise ###################
    
    if np.sum(incorrect_predictions)==0:
        print("no mistakes in this df")
    else:
        return incorrect_indices, predictions

---
## `print_mistake_k(int, df, numpy_array, numpy_array)`

This function has been directly obtained from the provided jupyter notebook in order to visualize some of the mistakes made by the model. No further modifications have been made.

In [ ]:
def print_mistake_k(k, df, mistake_indices, predictions):
    print(df.iloc[mistake_indices[k]].question1)
    print(df.iloc[mistake_indices[k]].question2)
    print("true class:", df.iloc[mistake_indices[k]].is_duplicate)
    print("prediction:", predictions[mistake_indices[k]])
    print()